In [1]:
# Kill all processes on the GPU
!fuser -v /dev/nvidia* -k


In [2]:
# Check the GPU status
!nvidia-smi


Thu Nov 26 03:46:59 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   73C    P8             15W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# Libraries


In [3]:
%%capture
!uv pip uninstall torchao torchaudio torchvision -y
!uv pip install \
    "transformers==4.53.3" \
    "peft==0.17.1" \
    "trl" \
    "accelerate" \
    "bitsandbytes" \
    "wandb" \
    "evaluate"


In [4]:
import os
import math
import random
from datetime import datetime

import torch
import wandb
import evaluate
from transformers import (
    XLMRobertaForQuestionAnswering,
    XLMRobertaForMaskedLM,
    DataCollatorForLanguageModeling, 
    DataCollatorWithPadding,
    AutoTokenizer,
    Trainer, 
    TrainingArguments,
    EarlyStoppingCallback,
    TrainerCallback,
)
from peft import (
    LoraConfig, 
    get_peft_model,
    PeftModel,
)
from datasets import load_dataset, Dataset


# Utilities


In [5]:
def load_train_val_datasets(
    task: str,  # 'wikipedia' | 'squad'
    lang: str,  # wikipedia: 'en' | '<any_lang>', squad: 'en' only
    train_size: int, 
    val_size: int,
) -> tuple[Dataset, Dataset]:
    # Validate that if the task is 'squad', the language must be 'en'
    if task == 'squad':
        assert lang == 'en', "SQuAD is English-only."
    
    # Define dataset configurations for each task
    data_configs = {
        'wikipedia': {
            'data_id': 'wikimedia/wikipedia',
            'data_dir': f'20231101.{lang}',
            'train_split': 'train',
            'val_split': 'train',
        },
        'squad': {
            'data_id': 'rajpurkar/squad',
            'data_dir': None,
            'train_split': 'train',
            'val_split': 'validation',
        },
    }
    
    # Validate that the specified task is supported
    assert task in data_configs, (
        f"Unsupported task: {task}. "
        f"Supported tasks: {list(data_configs.keys())}"
    )

    # Set up Hugging Face dataset configuration
    data_id = data_configs[task]['data_id']
    data_dir = data_configs[task]['data_dir']
    train_split = data_configs[task]['train_split']
    val_split = data_configs[task]['val_split']

    if train_split == val_split:
        # If the train and validation splits are the same, we need to sample from the same dataset stream
        dataset_stream = load_dataset(
            data_id,
            data_dir=data_dir,
            split=train_split,
            streaming=True,
        )

        train_data = []
        val_data = []

        for i, example in enumerate(dataset_stream):
            if i < train_size:
                train_data.append(example)
            elif i < train_size + val_size:
                val_data.append(example)
            else:
                break

    else:
        # If the train and validation splits are different, we can sample from each split separately
        def sample_split(split, size):
            dataset_stream = load_dataset(
                data_id,
                data_dir=data_dir,
                split=split,
                streaming=True,
            )

            data = []
            for i, example in enumerate(dataset_stream):
                if i >= size:
                    break
                data.append(example)
            return data

        train_data = sample_split(train_split, train_size)
        val_data = sample_split(val_split, val_size)

    return (
        Dataset.from_list(train_data),
        Dataset.from_list(val_data),
    )


# Configuration


In [6]:
# Run configuration
TASK = 'wikipedia'  # 'wikipedia' | 'squad'
LANG = 'en'         # wikipedia: 'en' | '<any_lang>', squad: 'en' only
SEED = 1337
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# Data configuration
MLM_PROB = 0.15
# --- 0.8K (80%:20%) ---
TRAIN_SIZE = 800
VAL_SIZE = 200
# --- 1.5K (80%:20%) ---
# TRAIN_SIZE = 1500
# VAL_SIZE = 375
# --- 2K (80%:20%) ---
# TRAIN_SIZE = 2_000
# VAL_SIZE = 500
# --- 3K (80%:20%) ---
# TRAIN_SIZE = 3_000
# VAL_SIZE = 750
# --- 3.5K (80%:20%) ---
# TRAIN_SIZE = 3_500
# VAL_SIZE = 875
# --- 4K (80%:20%) ---
# TRAIN_SIZE = 4_000
# VAL_SIZE = 1_000
# --- 15K (80%:20%) ---
# TRAIN_SIZE = 15_000
# VAL_SIZE = 3_750

# Model configuration
MODEL_ID = 'FacebookAI/xlm-roberta-base'
MODEL_NAME = 'XLM-R-Base'
# --- New training ---
RESUME_MODEL_ID = None
RESUME_CKPT_STEP = None
# --- Resume training ---
# RESUME_MODEL_ID = 'alxxtexxr/XLM-R-Base-wikipedia-en-0.8K-s123-LoRA-v261125234455'
# RESUME_CKPT_STEP = 744

# LoRA configuration
LORA_RANK = 8
LORA_ALPHA = 8
LORA_DROPOUT = 0.1
LORA_TARGET_MODULES = [
    # Not 'all-linear', since we exclude 'qa_outputs' for question answering tasks
    'query', 
    'key', 
    'value', 
    # Not all 'dense' layers, since we exclude 'lm_head.dense' for masked language modeling
    'intermediate.dense',
    'output.dense',
]

# Training configuration
MINI_BATCH_SIZE = 4
GRAD_ACCUM_STEPS = 4
LR = 1e-4
NUM_EPOCHS = 10
EARLY_STOPPING_PATIENCE = None
# --- wikipedia-en-0.8K ---
WARMUP_STEPS = 124
EVAL_STEPS = 62
SAVE_STEPS = 62
LOGGING_STEPS = 31
# --- wikipedia-tr-1.5K ---
# WARMUP_STEPS = 133
# EVAL_STEPS = 67
# SAVE_STEPS = 67
# LOGGING_STEPS = 33
# --- wikipedia, 2~4K ---
# WARMUP_STEPS = 50
# EVAL_STEPS = 100
# SAVE_STEPS = 100
# LOGGING_STEPS = 50
# --- squad, 15K ---
# WARMUP_STEPS = 100
# EVAL_STEPS = 300
# SAVE_STEPS = 300
# LOGGING_STEPS = 100


In [7]:
# Resume training configuration
resume_from_checkpoint = None
if RESUME_MODEL_ID:
    model_name = RESUME_MODEL_ID
    run_name = model_name.split('/')[-1]
    hub_model_id = RESUME_MODEL_ID
    
    from huggingface_hub import snapshot_download
    snapshot_download(repo_id=hub_model_id, local_dir=model_name)
    
    if RESUME_CKPT_STEP:
        resume_from_checkpoint = f"{hub_model_id}/checkpoint-{RESUME_CKPT_STEP}"
        # Ensure the checkpoint exists
        assert os.path.exists(resume_from_checkpoint), f"Checkpoint {resume_from_checkpoint} does not exist."
else:
    run_name = f'{MODEL_NAME}-{TASK}-{LANG}-{TRAIN_SIZE/1000:g}K-s{SEED}-LoRA-v{datetime.now().strftime("%y%m%d%H%M%S")}'
    
    # Automatically retrieve the username of the currently logged-in user
    from huggingface_hub import HfApi
    user_info = HfApi().whoami()
    username = user_info["name"]

    hub_model_id = f"{username}/{run_name}"
base_hub_model_id, version = hub_model_id.rsplit('-v', 1)
hub_merged_model_id = f'{base_hub_model_id}-mrg-v{version}'

print("Resume from checkpoint:", resume_from_checkpoint)
print("Model name:", MODEL_NAME)
print("Run name:", run_name)
print("Hub model ID:", hub_model_id)
print("Hub merged model ID:", hub_merged_model_id)


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Resume from checkpoint: None
Model name: XLM-R-Base
Run name: XLM-R-Base-wikipedia-en-0.8K-s1337-LoRA-v261126034712
Hub model ID: alxxtexxr/XLM-R-Base-wikipedia-en-0.8K-s1337-LoRA-v261126034712
Hub merged model ID: alxxtexxr/XLM-R-Base-wikipedia-en-0.8K-s1337-LoRA-mrg-v261126034712


# Model


In [8]:
# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# Determine the appropriate model classes based on the task
if TASK == 'squad':
    model_cls = XLMRobertaForQuestionAnswering
    task_type = 'QUESTION_ANS'
    data_collator = DataCollatorWithPadding(tokenizer)
    label_names = ['start_positions', 'end_positions']
else:
    model_cls = XLMRobertaForMaskedLM
    task_type = 'FEATURE_EXTRACTION'
    data_collator = DataCollatorForLanguageModeling(
        tokenizer=tokenizer,
        mlm=True,
        mlm_probability=MLM_PROB,
    )
    label_names = ['labels']

# Load the model
model = model_cls.from_pretrained(MODEL_ID)


Some weights of the model checkpoint at FacebookAI/xlm-roberta-base were not used when initializing XLMRobertaForMaskedLM: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing XLMRobertaForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing XLMRobertaForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [9]:
# Sanity check
# for n, p in model.named_parameters():
#     print(n, p.requires_grad)


In [10]:
if resume_from_checkpoint:
    # Load the LoRA adapter from the checkpoint and ensure it's in training mode
    model = PeftModel.from_pretrained(model, resume_from_checkpoint)
    model.train()                   # Ensure the resumed model is in training mode
    model.enable_adapter_layers()   # Explicitly unfreeze LoRA weights
else:
    # Set up a LoRA configuration and apply it to the model
    lora_config = LoraConfig(
        r=LORA_RANK,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        bias='none',
        task_type=task_type,
        target_modules=LORA_TARGET_MODULES,
    )
    model: PeftModel = get_peft_model(model, lora_config)

model = model.to(DEVICE)

model.print_trainable_parameters()
print("device:", model.device)


trainable params: 1,327,104 || all params: 279,622,290 || trainable%: 0.4746
device: cuda:0


In [11]:
# Sanity check
# for n, p in model.named_parameters():
#     if p.requires_grad:
#         print(n, p.requires_grad)


# Data


In [12]:
# Load the dataset
train_dataset, val_dataset = load_train_val_datasets(
    task=TASK, 
    lang=LANG,
    train_size=TRAIN_SIZE, 
    val_size=VAL_SIZE,
)

print("Training dataset:")
print(train_dataset)
print()
print("Validation dataset:")
print(val_dataset)


Resolving data files:   0%|          | 0/41 [00:00<?, ?it/s]

Training dataset:
Dataset({
    features: ['id', 'url', 'title', 'text'],
    num_rows: 800
})

Validation dataset:
Dataset({
    features: ['id', 'url', 'title', 'text'],
    num_rows: 200
})


In [13]:
# Preprocess the dataset
WIKI_MAX_LENGTH = 510  # 510 + 2 (BOS + EOS) = 512, matching XLM-R's max_position_embeddings
NUM_CHUNKS_PER_WIKI_ARTICLE = 3

# Bookkeeping data for string-level SQuAD evaluation
#
# IMPORTANT: datasets.map(batched=True) invokes preprocess_squad once per
# BATCH (default batch_size=1000), so the bookkeeping must be ACCUMULATED
# across batches. Overwriting it here would leave only the LAST batch's
# features, which is shorter than the full eval predictions and crashes
# compute_metrics_squad with IndexError: list index out of range.
_squad_eval_bookkeeping = {
    'feature_example_ids': [],
    'feature_offset_mappings': [],
    'feature_contexts': [],
    'references': [],
}

if TASK == 'squad':

    def preprocess_squad(examples, save_bookkeeping=False):
        tokenized = tokenizer(
            examples['question'],
            examples['context'],
            truncation='only_second', # Only truncate the context
            max_length=384,
            stride=128,
            return_overflowing_tokens=True,
            return_offsets_mapping=True,
            padding=False,
        )

        # Each generated window points back to its original SQuAD example.
        sample_mapping = tokenized.pop('overflow_to_sample_mapping') # Global index

        # Save offsets separately because the model does not need them.
        offset_mapping = tokenized.pop('offset_mapping') # Local index

        start_positions = []
        end_positions = []

        # Indices of windows that contain the complete answer.
        keep_indices = []

        for feature_idx, offsets in enumerate(offset_mapping):

            # Which original example produced this window?
            sample_idx = sample_mapping[feature_idx]

            answer = examples['answers'][sample_idx]

            answer_start_char = answer['answer_start'][0]
            answer_text = answer['text'][0]
            answer_end_char = (
                answer_start_char + len(answer_text)
            )

            # sequence_ids tells us which tokens belong to:
            # 0 = question
            # 1 = context
            # None = special tokens
            sequence_ids = tokenized.sequence_ids(feature_idx)

            # Find the first and last context tokens in this window.
            context_token_indices = [
                idx
                for idx, seq_id in enumerate(sequence_ids)
                if seq_id == 1
            ]

            if not context_token_indices:
                continue

            context_start = context_token_indices[0]
            context_end = context_token_indices[-1]

            # Check whether the COMPLETE answer is inside this window.
            #
            # If the answer starts before the first context token,
            # or ends after the last context token, this window
            # does not contain the complete answer.
            if offsets[context_start][0] > answer_start_char:
                continue

            if offsets[context_end][1] < answer_end_char:
                continue

            # Find the token containing the answer start.
            token_start = context_start

            while (
                token_start <= context_end
                and offsets[token_start][1] <= answer_start_char
            ):
                token_start += 1

            # Find the token containing the answer end.
            token_end = context_end

            while (
                token_end >= context_start
                and offsets[token_end][0] >= answer_end_char
            ):
                token_end -= 1

            # Safety check.
            if token_start > context_end or token_end < context_start:
                continue

            # Keep this window.
            keep_indices.append(feature_idx)
            start_positions.append(token_start)
            end_positions.append(token_end)

        # Keep only windows containing the complete answer.
        tokenized = {
            key: [
                values[i]
                for i in keep_indices
            ]
            for key, values in tokenized.items()
        }

        tokenized['start_positions'] = start_positions
        tokenized['end_positions'] = end_positions

        # Save bookkeeping data for string-level SQuAD evaluation
        if save_bookkeeping:
            # Extend (do NOT overwrite): .map() calls this function once per
            # batch, and the mapped dataset concatenates each batch's features
            # in call order, so appending keeps bookkeeping aligned with the
            # final dataset rows.
            _squad_eval_bookkeeping['feature_example_ids'].extend(
                examples['id'][sample_mapping[i]] for i in keep_indices
            )
            _squad_eval_bookkeeping['feature_offset_mappings'].extend(
                offset_mapping[i] for i in keep_indices
            )
            _squad_eval_bookkeeping['feature_contexts'].extend(
                examples['context'][sample_mapping[i]] for i in keep_indices
            )

            # Build references (one per unique example), deduped across batches
            seen = {
                ref['id']
                for ref in _squad_eval_bookkeeping['references']
            }
            for i in keep_indices:
                ex_id = examples['id'][sample_mapping[i]]
                if ex_id not in seen:
                    seen.add(ex_id)
                    _squad_eval_bookkeeping['references'].append({
                        'id': ex_id,
                        'answers': examples['answers'][sample_mapping[i]],
                    })

        return tokenized


    train_dataset = train_dataset.map(
        preprocess_squad,
        batched=True,
        remove_columns=train_dataset.column_names,
    )

    val_dataset = val_dataset.map(
        lambda examples: preprocess_squad(examples, save_bookkeeping=True),
        batched=True,
        remove_columns=val_dataset.column_names,
        # The bookkeeping is a SIDE EFFECT of this map call: never serve it
        # from the datasets cache, otherwise the function is skipped and the
        # bookkeeping stays empty/stale.
        load_from_cache_file=False,
    )

else:

    def tokenize_text(examples, indices):
        rng = random.Random(SEED)

        all_input_ids = []
        all_attention_masks = []
        all_labels = []

        for text, article_idx in zip(examples['text'], indices):

            # Tokenize the entire article without truncation.
            # Special tokens are added after chunking.
            article_tokens = tokenizer(
                text,
                truncation=False,
                add_special_tokens=False,
            )['input_ids']

            # Create non-overlapping 512-token chunks.
            chunks = [
                article_tokens[i:i + WIKI_MAX_LENGTH]
                for i in range(
                    0,
                    len(article_tokens),
                    WIKI_MAX_LENGTH,
                )
            ]

            # Keep only complete 512-token chunks.
            chunks = [
                chunk
                for chunk in chunks
                if len(chunk) == WIKI_MAX_LENGTH
            ]

            # Select up to N chunks randomly.
            num_chunks = min(
                NUM_CHUNKS_PER_WIKI_ARTICLE,
                len(chunks),
            )

            if num_chunks == 0:
                continue

            # Use a deterministic per-article RNG.
            article_rng = random.Random(
                SEED + article_idx
            )

            selected_chunks = article_rng.sample(
                chunks,
                num_chunks,
            )

            for chunk in selected_chunks:

                # Add special tokens in the same way
                # the XLM-R tokenizer normally does.
                input_ids = (
                    [tokenizer.bos_token_id]
                    + chunk
                    + [tokenizer.eos_token_id]
                )
                attention_mask = [1] * len(input_ids)

                all_input_ids.append(input_ids)
                all_attention_masks.append(attention_mask)
                all_labels.append(input_ids.copy())

        return {
            'input_ids': all_input_ids,
            'attention_mask': all_attention_masks,
            'labels': all_labels,
        }

    train_dataset = train_dataset.map(
        tokenize_text,
        batched=True,
        with_indices=True,
        remove_columns=train_dataset.column_names,
    )

    val_dataset = val_dataset.map(
        tokenize_text,
        batched=True,
        with_indices=True,
        remove_columns=val_dataset.column_names,
    )


Map:   0%|          | 0/800 [00:00<?, ? examples/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (10426 > 512). Running this sequence through the model will result in indexing errors


Map:   0%|          | 0/200 [00:00<?, ? examples/s]

In [14]:
print("Total preprocessed training samples:", len(train_dataset))
print("Total preprocessed validation samples:", len(val_dataset))


Total preprocessed training samples: 1971
Total preprocessed validation samples: 527


# Training


In [15]:
# Calculate steps per epoch and max steps
steps_per_epoch = math.ceil(len(train_dataset) / (MINI_BATCH_SIZE * GRAD_ACCUM_STEPS))
max_steps = steps_per_epoch * NUM_EPOCHS
print("Steps per epoch:", steps_per_epoch)
print("Max steps:", max_steps)


Steps per epoch: 124
Max steps: 1240


# Diagnostics

Two extra metrics are logged next to the usual training/eval metrics:

- `eval/centered_moving_average_loss`: simple moving average of the
  `EVAL_LOSS_MA_WINDOW` validation losses around each eval, so every value
  describes the eval at its own step and the curve is not shifted. A centered
  window needs `EVAL_LOSS_MA_WINDOW // 2` later evals, so a value is written once
  its window closes - half a window after the eval it describes - and the last
  `EVAL_LOSS_MA_WINDOW // 2` values are written when training ends. Windows are
  clipped at the ends of the run (the first and last values average fewer
  points), so there is exactly one value per eval.
- `lora_dw_frobenius/*`: Frobenius norm of the effective LoRA update
  `ΔW = (lora_alpha / r) · B @ A`, aggregated (`total`, `mean`, `max`) and, when
  `LOG_LORA_DW_PER_LAYER` is on, one value per adapted module. These are logged
  directly to wandb and TensorBoard, because `rewrite_logs` prefixes every key
  that is not `eval_*`/`test_*` with `train/`; bypassing it keeps the keys
  top-level. As a result they do not appear in `trainer.state.log_history`.

Both are produced by wrapping `Trainer.log`: a callback cannot do it, because
`Trainer._evaluate` logs the eval metrics *before* `on_evaluate` runs, and the
reporting callbacks (wandb, TensorBoard) are registered before user callbacks.
The centered average cannot ride on the `Trainer.log` row - it belongs to an
earlier step, while `Trainer.log` only writes at the current one - so it goes
straight to wandb and TensorBoard at the step it describes.


In [16]:
# Diagnostics configuration
EVAL_LOSS_MA_WINDOW = 3       # Odd, so the centered window spans [i - 1, i + 1] for 3
LOG_LORA_DW_PER_LAYER = True  # Also log one Frobenius norm per LoRA adapter (~60 metrics)


def lora_delta_frobenius_norms(model: torch.nn.Module) -> dict[str, float]:
    """Return ||ΔW||_F per LoRA adapter, with ΔW = (lora_alpha / r) · B @ A.

    The scaling factor is the one PEFT uses when adding the update to the frozen
    weight, so runs with different lora_alpha / r stay comparable. Computed in
    float32 regardless of the mixed-precision training dtype.
    """
    norms = {}
    for name, module in model.named_modules():
        if not hasattr(module, 'lora_A') or not hasattr(module, 'lora_B'):
            continue
        for adapter_name in module.lora_A.keys():
            A = module.lora_A[adapter_name].weight.detach().float()
            B = module.lora_B[adapter_name].weight.detach().float()
            scaling = module.scaling[adapter_name]  # lora_alpha / r
            delta = (B @ A) * scaling
            norms[f'{name}.{adapter_name}'] = delta.norm(p='fro').item()
    return norms


def _tensorboard_writer(trainer: Trainer):
    """Return the TensorBoard summary writer, or None when TensorBoard is off."""
    try:
        from transformers.integrations import TensorBoardCallback
    except ImportError:
        return None
    for callback in trainer.callback_handler.callbacks:
        if isinstance(callback, TensorBoardCallback):
            return getattr(callback, 'tb_writer', None)
    return None


def _log_scalars_at(trainer: Trainer, metrics: dict[str, float], x_step: int) -> None:
    """Log scalars at an explicit x-axis step, bypassing `Trainer.log`.

    `Trainer.log` can only write at the current `global_step`, which is the wrong
    position for a value that describes an earlier eval. wandb charts every metric
    against the `train/global_step` field of the row (the x-axis `WandbCallback`
    defines), so passing an older step there moves the point back where it belongs;
    TensorBoard takes the step as an argument. The wandb step counter itself is left
    to auto-increment, exactly as `WandbCallback.on_log` does.
    """
    if wandb.run is not None and trainer.is_world_process_zero():
        wandb.log({**metrics, 'train/global_step': x_step})

    writer = _tensorboard_writer(trainer)
    if writer is not None:
        for key, value in metrics.items():
            writer.add_scalar(key, value, x_step)


def _log_lora_dw_norms(trainer: Trainer, norms: dict[str, float], per_layer: bool) -> None:
    """Log the ΔW norms straight to wandb and TensorBoard, bypassing `Trainer.log`.

    They have to bypass it: `transformers.integrations.integration_utils.rewrite_logs`
    prefixes every key that does not start with `eval_`/`test_` with `train/`, so
    the log stream would surface them as `train/lora_dw_frobenius/*`.
    """
    values = list(norms.values())
    metrics = {
        'lora_dw_frobenius/total': math.sqrt(sum(v * v for v in values)),
        'lora_dw_frobenius/mean': sum(values) / len(values),
        'lora_dw_frobenius/max': max(values),
    }
    if per_layer:
        for name, value in norms.items():
            metrics[f'lora_dw_frobenius/layer/{name}'] = value

    _log_scalars_at(trainer, metrics, trainer.state.global_step)


def _centered_window_mean(values: list[float], center: int, half_window: int) -> float:
    """Mean of `values` over `[center - half_window, center + half_window]`.

    The window is clipped to the list, so the first and last values average fewer
    points instead of being dropped: one centered value per eval, always.
    """
    lo = max(0, center - half_window)
    hi = min(len(values), center + half_window + 1)
    window = values[lo:hi]
    return sum(window) / len(window)


def add_lora_diagnostics(
    trainer: Trainer,
    ma_window: int = EVAL_LOSS_MA_WINDOW,
    per_layer: bool = LOG_LORA_DW_PER_LAYER,
) -> Trainer:
    """Log a centered moving average of eval_loss and the Frobenius norm of LoRA ΔW.

    The average is written once its window closes, at the step of the eval it
    describes, as `eval/centered_moving_average_loss` (see `_log_scalars_at`). The
    ΔW norms are written after each row, directly to wandb and TensorBoard (see
    `_log_lora_dw_norms`).
    """
    if ma_window % 2 != 1:
        raise ValueError(f'ma_window must be odd for a centered average, got {ma_window}')
    half_window = ma_window // 2

    original_log = trainer.log
    eval_losses: list[float] = []
    eval_loss_steps: list[int] = []
    next_centered = 0  # index of the first eval that has no centered value yet

    def log_centered_upto(last_index: int) -> None:
        """Write centered values for evals up to `last_index`, whose windows have closed."""
        nonlocal next_centered
        while next_centered <= last_index:
            _log_scalars_at(
                trainer,
                {
                    'eval/centered_moving_average_loss': _centered_window_mean(
                        eval_losses, next_centered, half_window
                    )
                },
                eval_loss_steps[next_centered],
            )
            next_centered += 1

    def log_with_diagnostics(logs: dict[str, float], *args, **kwargs):
        is_eval = bool(logs) and 'eval_loss' in logs

        if is_eval:
            eval_losses.append(float(logs['eval_loss']))
            eval_loss_steps.append(int(trainer.state.global_step))

        # Write the row first, so the TensorBoard callback has created its writer
        result = original_log(logs, *args, **kwargs)

        # The window centered on eval i closes when eval i + half_window lands
        if is_eval:
            log_centered_upto(len(eval_losses) - 1 - half_window)

        if logs:
            model = getattr(trainer, 'model', None)
            norms = lora_delta_frobenius_norms(model) if model is not None else {}
            if norms:
                _log_lora_dw_norms(trainer, norms, per_layer)
        return result

    class CenteredTailFlush(TrainerCallback):
        """Write the centered values whose window only closes after training ends."""

        def on_train_end(self, args, state, control, **kwargs):
            log_centered_upto(len(eval_losses) - 1)

    trainer.log = log_with_diagnostics
    # First in the list, so it runs before any reporting callback that closes the
    # run (`WandbCallback.on_train_end` did call `wandb.finish()` in older
    # transformers); logs sent to a finished run are dropped.
    trainer.callback_handler.callbacks.insert(0, CenteredTailFlush())
    return trainer


In [17]:
# Initialize wandb
wandb.init(
    project="legamex",
    name=run_name,
)

# Load the SQuAD metric (string-level Exact Match and F1)
squad_metric = evaluate.load('squad')

# Compute metrics for SQuAD (string-level Exact Match and F1 via HF evaluate)
def compute_metrics_squad(eval_pred):
    start_logits, end_logits = eval_pred.predictions

    # Defensive check: bookkeeping must be feature-for-feature aligned with
    # the eval predictions. A mismatch means the preprocessing cell that
    # builds _squad_eval_bookkeeping did not run (or ran on a different
    # val set) - re-run it before training.
    n_features = len(start_logits)
    n_bookkeeping = len(_squad_eval_bookkeeping['feature_offset_mappings'])
    if n_features != n_bookkeeping:
        raise RuntimeError(
            f'Eval produced {n_features} features but the SQuAD bookkeeping '
            f'only has {n_bookkeeping}. Re-run the dataset preprocessing '
            f'cell so the bookkeeping is rebuilt for the full validation set.'
        )

    # For each feature, find the best span and reconstruct predicted text
    feature_predictions = []
    for feat_idx in range(len(start_logits)):
        s_log = start_logits[feat_idx]
        e_log = end_logits[feat_idx]
        offsets = _squad_eval_bookkeeping['feature_offset_mappings'][feat_idx]
        context = _squad_eval_bookkeeping['feature_contexts'][feat_idx]
        ex_id = _squad_eval_bookkeeping['feature_example_ids'][feat_idx]

        # Identify context tokens (non-zero offset range)
        context_tokens = [
            tok for tok, (start, end) in enumerate(offsets)
            if start != 0 or end != 0
        ]

        if not context_tokens:
            feature_predictions.append({
                'id': ex_id,
                'prediction_text': '',
                'score': -float('inf'),
            })
            continue

        # Find best span using top-k candidates
        n_best = 20
        k = min(n_best, len(context_tokens))
        top_starts = sorted(context_tokens, key=lambda t: s_log[t], reverse=True)[:k]
        top_ends = sorted(context_tokens, key=lambda t: e_log[t], reverse=True)[:k]

        best_score = -float('inf')
        best_start = context_tokens[0]
        best_end = context_tokens[0]

        for start_tok in top_starts:
            for end_tok in top_ends:
                if end_tok < start_tok:
                    continue
                score = float(s_log[start_tok] + e_log[end_tok])
                if score > best_score:
                    best_score = score
                    best_start = start_tok
                    best_end = end_tok

        # Reconstruct text
        start_char = offsets[best_start][0]
        end_char = offsets[best_end][1]
        pred_text = context[start_char:end_char]

        feature_predictions.append({
            'id': ex_id,
            'prediction_text': pred_text,
            'score': best_score,
        })

    # Group by example_id, pick best prediction per example
    example_predictions = {}
    for pred in feature_predictions:
        ex_id = pred['id']
        if ex_id not in example_predictions or pred['score'] > example_predictions[ex_id]['score']:
            example_predictions[ex_id] = pred

    predictions = [
        {'id': ex_id, 'prediction_text': pred['prediction_text']}
        for ex_id, pred in example_predictions.items()
    ]

    references = _squad_eval_bookkeeping['references']

    if not predictions or not references:
        return {'f1': 0.0, 'exact_match': 0.0}

    metrics = squad_metric.compute(predictions=predictions, references=references) or {
        'f1': 0.0,
        'exact_match': 0.0,
    }

    return {
        'f1': float(metrics.get('f1', 0.0)),
        'exact_match': float(metrics.get('exact_match', 0.0)),
    }

# Set up a trainer
training_args = TrainingArguments(
    # Training arguments
    seed=SEED,
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    per_device_train_batch_size=MINI_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    max_steps = max_steps,
    warmup_steps = WARMUP_STEPS,
    learning_rate=LR,
    lr_scheduler_type='cosine',
    optim='adamw_8bit',
    max_grad_norm=1.0,
    weight_decay=0.01,
    
    # Validation arguments
    eval_strategy='steps',
    eval_steps=EVAL_STEPS,
    
    # Logging arguments
    logging_strategy='steps',
    logging_steps=LOGGING_STEPS,
    # logging_first_step=True,
    report_to=['tensorboard', 'wandb'],
    
    # Saving arguments
    save_strategy='steps',
    save_steps=SAVE_STEPS,
    # save_total_limit=5, # 1 best + 4 recent checkpoints. WARN: It doesn't work
    
    # With load_best_model_at_end=True, your save_strategy will be ignored and default to eval_strategy.
    # So you will find one checkpoint at the end of each epoch.
    # https://discuss.huggingface.co/t/trainer-not-saving-after-save-steps/5464
    load_best_model_at_end=True,
    metric_for_best_model='f1' if TASK == 'squad' else 'eval_loss',
    greater_is_better=True if TASK == 'squad' else False,

    # run_name=run_name,
    output_dir=run_name,
    hub_model_id=hub_model_id,
    push_to_hub=True,
    hub_strategy='all_checkpoints',
    hub_always_push=True,
)
trainer = Trainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    args=training_args,
    compute_metrics=compute_metrics_squad if TASK == 'squad' else None,
    # label_names=['labels'],
    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=EARLY_STOPPING_PATIENCE,
            # early_stopping_threshold = 0.001,
        )
    ] if EARLY_STOPPING_PATIENCE is not None else [],
)
trainer.label_names = label_names

# Extra logging: moving-average val loss and Frobenius norm of LoRA ΔW
trainer = add_lora_diagnostics(trainer)


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: alimtegar to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


No label_names provided for model class `PeftModelForFeatureExtraction`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


In [18]:
# Start training
trainer_stats = trainer.train(resume_from_checkpoint=resume_from_checkpoint)


wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.


Step,Training Loss,Validation Loss
62,1.641300,1.487812
124,1.600200,1.469394
186,1.636600,1.481493
248,1.547600,1.490916
310,1.600500,1.495055
372,1.578800,1.488412
434,1.599300,1.466301
496,1.551800,1.489760
558,1.617500,1.484353
620,1.557700,1.454695


# Merging and Uploading


In [19]:
# If the task is SQuAD, upload the merged model and tokenizer
if TASK == 'squad':
    # After the training finishes, merge the LoRA into the base model and save everything
    model = model.eval() # Good practice
    merged_model = model.merge_and_unload()

    # Upload the merged model to Hugging Face
    merged_model.push_to_hub(hub_merged_model_id)
    tokenizer.push_to_hub(hub_merged_model_id)

    print(f"Merged model uploaded to: https://huggingface.co/{hub_merged_model_id}")
